# 08. Explainable AI (XAI) with SHAP: Heart Disease Screening Module

**Phase**: Heart Disease — Phase 8  
**Document Reference**: `docs/HEART_SHAP_EXPLAINABILITY.md`  
**Parent Artifacts**: `artifacts/heart/experiments/`, `artifacts/heart/threshold/`, `artifacts/heart/calibration/`  
**New Phase 8 Artifacts**: `artifacts/heart/shap/`  
**Dataset**: Cleveland Heart Disease (`datasets/raw/heart/processed.cleveland.data`)  
**Strict Phase Boundary**: Model explainability only. No production FastAPI bundling, no frontend implementation, no clinical diagnosis claims.

---

### Section 1: Objective & Background
In clinical cardiovascular risk screening, machine learning models cannot function as black boxes. Healthcare professionals, clinical auditors, and patients need transparent, auditable answers to:
> **"Which input features contributed to the model's prediction, and in what direction?"**

This notebook establishes an Explainable AI (XAI) layer using **SHAP (SHapley Additive exPlanations)** based on cooperative game theory (Shapley values). The pipeline implemented here corresponds to:

$$\begin{matrix}
\text{Patient Measurements (13 Features)} \\
\downarrow \\
\text{Leakage-Safe Preprocessor (One-Hot + Robust Scaling)} \\
\downarrow \\
\text{Final Model (Logistic Regression L1 / Tree Ensembles)} \\
\downarrow \\
\text{Model Output Space (Log-Odds / Probability)} \\
\downarrow \\
\text{SHAP Explainer (LinearExplainer / TreeExplainer)} \\
\downarrow \\
\text{Feature-Level Attributions (Additive Shapley Values)} \\
\downarrow \\
\text{Auditable Clinical Attribution Explanation}
\end{matrix}$$

> [!IMPORTANT]
> **Safety & Research Language Constraints**:
> - SHAP values explain the mathematical contribution of an input feature toward the **model's numerical prediction**.
> - SHAP does **not** establish medical causation. It does not prove that a feature clinically caused coronary artery disease.
> - A model risk score is **not** a clinical medical diagnosis.


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

# Add ml-service to path
sys.path.append(os.path.abspath("../ml-service"))

from app.heart_preprocessing import (
    verify_raw_dataset_integrity,
    load_heart_raw_data,
    create_modeling_dataset,
    split_heart_data,
    get_heart_feature_names,
)
from app.heart_calibration import get_heart_finalist_configs
from app.heart_shap import (
    get_heart_feature_mapping,
    fit_model_and_preprocessor_for_shap,
    create_heart_shap_explainer,
    compute_shap_explanations,
    compute_global_feature_importance,
    explain_single_observation,
    generate_representative_local_explanations,
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

print("Libraries imported successfully.")
print(f"SHAP Version:        {shap.__version__}")
print(f"Reproducible Seed:   {RANDOM_STATE}")


### Section 2: Dataset Integrity & Train/Holdout Separation

We verify the cryptographic integrity of `processed.cleveland.data` and recreate the locked 80/20 train/test partition:
- **Training Cohort ($80\%$)**: 242 samples (111 cases, 131 controls).
- **Hold-out Cohort ($20\%$)**: 61 samples (28 cases, 33 controls).


In [ ]:
raw_path = os.path.abspath("../datasets/raw/heart/processed.cleveland.data")
is_valid, md5_hash, sha_hash, file_sz = verify_raw_dataset_integrity(raw_path)
print(f"Dataset Verification: Valid={is_valid}")
print(f"  MD5:    {md5_hash}")
print(f"  SHA256: {sha_hash}")
print(f"  Size:   {file_sz} bytes")
assert is_valid, "Raw dataset integrity check failed!"

df_raw = load_heart_raw_data(raw_path)
X, y = create_modeling_dataset(df_raw)
X_train, X_test, y_train, y_test = split_heart_data(X, y, test_size=0.20, random_state=RANDOM_STATE)
print(f"\nCohort Split:")
print(f"  Training cohort: {len(X_train)} samples ({y_train.sum()} positive, {(y_train == 0).sum()} negative)")
print(f"  Holdout cohort:  {len(X_test)} samples ({y_test.sum()} positive, {(y_test == 0).sum()} negative)")


### Section 3: Final Model Configuration & Calibration Context

Based on the cumulative evidence from Phases 5–7:
1. **Phase 5 (Discrimination)**:
   - **Logistic Regression (Robust L1, C=1.0)** achieved the highest 25-fold Repeated Stratified CV ROC-AUC ($0.9076 \pm 0.0506$) and holdout ROC-AUC of $0.9513$. L1 regularization enforced sparse feature selection (10 non-zero features).
2. **Phase 6 (Thresholds)**:
   - Logistic Regression established a robust screening cutoff at $t=0.31$ (Sensitivity $\ge 90.1\%$, Specificity $= 64.9\%$ on OOF; Holdout Sensitivity $= 92.9\%$, Specificity $= 78.8\%$).
3. **Phase 7 (Calibration)**:
   - Logistic Regression proved to be the **most well-calibrated native model** across the entire project: $\text{ECE} = 0.0383$, $\text{Brier} = 0.1188$, $\text{Log Loss} = 0.3848$, and Calibration Slope $= 1.0704$ (ideal is $1.00$).

Therefore, **Logistic Regression (Robust L1)** serves as the primary finalized model architecture, with **Random Forest** and **XGBoost** evaluated comparatively.

#### SHAP Output Target Definition:
- **Logistic Regression**: SHAP explains the model's **log-odds decision space** (`decision_function`):
  $$\text{base\_value} + \sum_{j=1}^{18} \phi_j(x) = \text{decision\_function}(x)$$
  Because $P(Y=1) = \frac{1}{1 + e^{-\text{decision\_function}(x)}}$ is strictly monotonic, positive SHAP values push predicted probability higher toward disease, and negative SHAP values pull predicted probability lower toward control.


In [ ]:
candidate_configs = get_heart_finalist_configs()
primary_cfg = candidate_configs[0]

# Fit primary model and preprocessor
est_lr, prep_lr, X_tr_lr, feat_names_lr = fit_model_and_preprocessor_for_shap(
    primary_cfg, X_train, y_train
)

# Feature mapping
mapping_lr = get_heart_feature_mapping("LogisticRegression", preprocessor=prep_lr)
print(f"Primary Architecture: {primary_cfg['candidate_name']}")
print(f"Transformed features ({len(feat_names_lr)}):")
for idx, fn in enumerate(feat_names_lr, 1):
    d_name = mapping_lr[mapping_lr["transformed_feature"] == fn]["display_name"].values[0]
    print(f"  {idx:2d}. {fn:<22} -> {d_name}")


### Section 4: SHAP Explainer Initialization & Additivity Verification

For Logistic Regression, we initialize `shap.LinearExplainer` with `shap.maskers.Independent(X_train_transformed)` covering the full training distribution ($N=242$).

We verify the foundational **Efficiency / Additivity Axiom** on the holdout test set:
$$\text{base\_value} + \sum_{j=1}^{18} \phi_j(x) \equiv \text{decision\_function}(x)$$


In [ ]:
explainer_lr = create_heart_shap_explainer(est_lr, X_tr_lr, "LogisticRegression")
exp_lr_test = compute_shap_explanations(explainer_lr, prep_lr, est_lr, X_test, "LogisticRegression")

# Verify Additivity Axiom
reconstructed = exp_lr_test["base_values"] + exp_lr_test["shap_values"].sum(axis=1)
actual_outputs = exp_lr_test["model_outputs"]
discrepancy = np.max(np.abs(reconstructed - actual_outputs))

print(f"SHAP Efficiency / Additivity Check:")
print(f"  Base value (expected log-odds): {exp_lr_test['base_values'][0]:.4f}")
print(f"  Max absolute discrepancy:       {discrepancy:.2e}")
assert discrepancy < 1e-10, "Additivity check failed!"
print("Exact mathematical additivity verified.")


### Section 5: Global Feature Importance Analysis

Global feature importance is measured by the mean absolute SHAP value across the evaluation cohort:
$$\text{Importance}_j = \frac{1}{N} \sum_{i=1}^N |\phi_{i, j}|$$
This quantifies the average magnitude of influence each feature exerts on the model's log-odds output.


In [ ]:
df_imp_lr = compute_global_feature_importance(
    exp_lr_test["shap_values"], feat_names_lr, mapping_lr
)
print("=== GLOBAL MODEL FEATURE CONTRIBUTIONS (LOGISTIC REGRESSION) ===")
display(df_imp_lr[["global_rank", "display_name", "original_feature", "mean_abs_shap", "pct_importance"]])

# Bar Plot
plt.figure(figsize=(10, 6.5))
top_feats = df_imp_lr.head(12)
colors = sns.color_palette("Blues_r", len(top_feats))
bars = plt.barh(range(len(top_feats)), top_feats["mean_abs_shap"][::-1], color=colors[::-1])
plt.yticks(range(len(top_feats)), top_feats["display_name"][::-1], fontsize=10)
plt.xlabel("Mean |SHAP Value| (Average Impact on Model Log-Odds Output)", fontsize=11, fontweight="bold")
plt.title("Heart Disease — Global Feature Importance (Logistic Regression L1)
Mean Absolute SHAP Value across Test Cohort (N=61)", fontsize=13, fontweight="bold")

for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.02, bar.get_y() + bar.get_height()/2, f"{w:.3f}", ha="left", va="center", fontsize=9)
    
plt.xlim(0, max(top_feats["mean_abs_shap"]) * 1.15)
plt.tight_layout()
plt.show()


### Section 6: Directional Impact — SHAP Beeswarm Summary

The beeswarm plot displays both feature importance and the direction of feature impact:
- Each dot represents a patient from the holdout cohort ($N=61$).
- **Color** indicates feature value (red = high value, blue = low value).
- **Position on the x-axis** indicates SHAP value (positive = pushes prediction toward disease, negative = pulls prediction toward control).


In [ ]:
fig, ax = plt.subplots(figsize=(11, 7.5))
exp_obj = shap.Explanation(
    values=exp_lr_test["shap_values"],
    base_values=exp_lr_test["base_values"],
    data=exp_lr_test["X_transformed"],
    feature_names=[mapping_lr[mapping_lr["transformed_feature"] == fn]["display_name"].values[0] for fn in feat_names_lr]
)
plt.sca(ax)
shap.plots.beeswarm(exp_obj, max_display=12, show=False)
plt.title("Heart Disease — Directional Feature Impact (Logistic Regression Beeswarm)
Hold-Out Test Cohort (N=61)", fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()


### Section 7: Multi-Model Architecture Comparison

To verify that feature attribution patterns reflect genuine dataset signals rather than model artifacts, we compare global feature rankings across **Logistic Regression**, **Random Forest**, and **XGBoost**.


In [ ]:
# Fit RF and XGBoost
rf_cfg = candidate_configs[1]
est_rf, prep_rf, X_tr_rf, feat_names_rf = fit_model_and_preprocessor_for_shap(rf_cfg, X_train, y_train)
explainer_rf = create_heart_shap_explainer(est_rf, X_tr_rf, "RandomForest")
exp_rf_test = compute_shap_explanations(explainer_rf, prep_rf, est_rf, X_test, "RandomForest")
mapping_tree = get_heart_feature_mapping("RandomForest", preprocessor=prep_rf)
df_imp_rf = compute_global_feature_importance(exp_rf_test["shap_values"], feat_names_rf, mapping_tree)

xgb_cfg = candidate_configs[2]
est_xgb, prep_xgb, X_tr_xgb, feat_names_xgb = fit_model_and_preprocessor_for_shap(xgb_cfg, X_train, y_train)
explainer_xgb = create_heart_shap_explainer(est_xgb, X_tr_xgb, "XGBoost")
exp_xgb_test = compute_shap_explanations(explainer_xgb, prep_xgb, est_xgb, X_test, "XGBoost")
df_imp_xgb = compute_global_feature_importance(exp_xgb_test["shap_values"], feat_names_xgb, mapping_tree)

# 3-Panel Comparison Plot
fig, axes = plt.subplots(1, 3, figsize=(20, 6.5))
models_info = [
    ("Logistic Regression (Robust L1)", df_imp_lr, "Blues_r", axes[0]),
    ("Random Forest (n=200, leaf=5)", df_imp_rf, "Greens_r", axes[1]),
    ("XGBoost (n=100, depth=2, lr=0.05)", df_imp_xgb, "Oranges_r", axes[2]),
]
for m_title, df_m, pal, ax in models_info:
    top_m = df_m.head(8)
    c_list = sns.color_palette(pal, len(top_m))
    ax.barh(range(len(top_m)), top_m["mean_abs_shap"][::-1], color=c_list[::-1])
    ax.set_yticks(range(len(top_m)))
    ax.set_yticklabels(top_m["display_name"][::-1], fontsize=10)
    ax.set_xlabel("Mean |SHAP Value|", fontsize=11, fontweight="bold")
    ax.set_title(m_title, fontsize=12, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.5)

plt.suptitle("Heart Disease Phase 8 — Global SHAP Feature Importance across Finalist Architectures", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


### Section 8: Local Patient Explanations (Hold-Out Test Cases)

We examine real representative patient observations from the holdout set ($N=61$) to audit individual decision logic:
- **True Positive**: Patient with verified CAD receiving a high estimated risk.
- **True Negative**: Control patient receiving a low estimated risk.
- **Borderline Case**: Patient receiving an intermediate probability close to 0.50.
- **Discordant Cases**: False Positive and False Negative observations.


In [ ]:
df_local_lr = generate_representative_local_explanations(
    explainer_lr, prep_lr, est_lr, X_test, y_test, model_family="LogisticRegression"
)
display_cols = ["case_type", "sample_index", "true_label", "model_estimated_probability", "top_positive_contributors", "top_negative_contributors"]
display(df_local_lr[display_cols])

# Local Waterfall Visualizations
fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))

# Case 1: True Positive
tp_idx = int(df_local_lr[df_local_lr["case_type"] == "High-Confidence True Positive"]["sample_index"].iloc[0])
test_pos_tp = list(X_test.index).index(tp_idx)
exp_tp = shap.Explanation(
    values=exp_lr_test["shap_values"][test_pos_tp],
    base_values=exp_lr_test["base_values"][test_pos_tp],
    data=exp_lr_test["X_transformed"][test_pos_tp],
    feature_names=[mapping_lr[mapping_lr["transformed_feature"] == fn]["display_name"].values[0] for fn in feat_names_lr]
)
plt.sca(axes[0])
shap.plots.waterfall(exp_tp, max_display=8, show=False)
axes[0].set_title(f"True Positive (Sample #{tp_idx}) | Estimated Prob = {df_local_lr[df_local_lr['case_type'] == 'High-Confidence True Positive']['model_estimated_probability'].iloc[0]:.2f}", fontsize=11, fontweight="bold")

# Case 2: True Negative
tn_idx = int(df_local_lr[df_local_lr["case_type"] == "High-Confidence True Negative"]["sample_index"].iloc[0])
test_pos_tn = list(X_test.index).index(tn_idx)
exp_tn = shap.Explanation(
    values=exp_lr_test["shap_values"][test_pos_tn],
    base_values=exp_lr_test["base_values"][test_pos_tn],
    data=exp_lr_test["X_transformed"][test_pos_tn],
    feature_names=[mapping_lr[mapping_lr["transformed_feature"] == fn]["display_name"].values[0] for fn in feat_names_lr]
)
plt.sca(axes[1])
shap.plots.waterfall(exp_tn, max_display=8, show=False)
axes[1].set_title(f"True Negative (Sample #{tn_idx}) | Estimated Prob = {df_local_lr[df_local_lr['case_type'] == 'High-Confidence True Negative']['model_estimated_probability'].iloc[0]:.2f}", fontsize=11, fontweight="bold")

plt.suptitle("Local Patient Waterfall Attributions (Holdout Test Cohort)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


### Section 9: Limitations, Safety Boundaries & Production Handoff

#### 1. Clinical Interpretation Boundaries:
- SHAP values quantify **model contributions**, not physiological causality. For example, a high contribution from `Fluoroscopy Vessels (ca)` reflects the statistical strength of vessel stenosis in this dataset; it does not replace angiographic interpretation.
- A model-estimated probability does not constitute a clinical diagnosis.

#### 2. Relationship to Probability Calibration:
- For Logistic Regression, SHAP values are computed in the linear log-odds space. Because Logistic Regression was natively calibrated in Phase 7, the log-odds directly map to well-calibrated probabilities via the standard sigmoid without post-hoc distortion.

#### 3. Handoff to Production Artifacts & Service Integration:
- In subsequent phases, the preprocessor, finalist model, and SHAP background masker will be packaged into a production bundle (`heart_model_bundle.joblib`).
- The explanation utility will power the FastAPI endpoint `/heart/predict` and the clinical user interface, delivering local feature attributions alongside model-estimated probabilities.
